### 01. Implementing modelling

First we want to create a simple linear regression to serve as our baseline.

In [5]:
# First we got to do some imports (I'm just grabbing the ones from 02 for now)
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression


# Random seed for reproducibility
np.random.seed(42)

df = pd.read_csv("data/ecowas_df_synthetic_full.csv")

In [6]:
df.columns

Index(['Unnamed: 0', 'country', 'year', 'disorder_Demonstrations',
       'disorder_Political violence',
       'disorder_Political violence; Demonstrations',
       'disorder_Strategic developments', 'event_Battles',
       'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
       'event_Strategic developments', 'event_Violence against civilians',
       'perpetrator_Civilians', 'perpetrator_External/Other forces',
       'perpetrator_Identity militia', 'perpetrator_Political militia',
       'perpetrator_Protesters', 'perpetrator_Rebel group',
       'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
       'target_External/Other forces', 'target_Identity militia',
       'target_Political militia', 'target_Protesters', 'target_Rebel group',
       'target_Rioters', 'target_State forces', 'fatalities', 'iso3_o',
       'iso3_d', 'distw_harmonic', 'distw_arithmetic', 'dist', 'distcap',
       'contig', 'diplo_disagreement', 'comlang_off', 'comlang

In [7]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

In [9]:
test = ["hello", "this", "is", "a", "test"]

test.remove("a")

#print(len(test)+1)

for i in range(0, len(test)+1, 2):
    print(i) 
    print(i+1)


0
1
2
3
4
5


### Log/Linear Regression on Economic/Gravity Data

This will serve as our primary baseline. If the machine learning model does not improve on this, then it does not add predictive power beyond simple Gravity/trade fundamentals

In [10]:
legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}

def baseline_linear(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016):
    '''
    The main function for creating our regression

    
    Input:
        df: pd.DataFrame -
            The main dataframe to work with.
        target_variable: str -
            The column which we are inferring for. 
        columns: list -
            A list of column elements that are included in the model. NOTE: Always include the basic Gravity columns, other we raise an error
        year_split: int -
            The year where we split between train and test. Defaults to 2016
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    


    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = LinearRegression()
    model.fit(X_train, y_train)


    # Finally we can get some workable metrics from this regression:
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample R²:", r2)



    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                                cov_kwds={"groups": groups},)

    print(ols.summary())


    return target_variable
    


In [11]:
columns = [
    "year",
    "gdp_o", "gdp_d",
    "distw_arithmetic",
    "contig",
    "comlang_off",
]

baseline_linear(df, "tradeflow_baci", columns)

Out-of-sample RMSE: 2.4179838553739543
Out-of-sample R²: 0.37193980677223815
                               OLS Regression Results                               
Dep. Variable:     tradeflow_baci_log_trade   R-squared:                       0.384
Model:                                  OLS   Adj. R-squared:                  0.383
Method:                       Least Squares   F-statistic:                     58.30
Date:                      Fri, 24 Apr 2026   Prob (F-statistic):           6.82e-29
Time:                              10:39:29   Log-Likelihood:                -8307.6
No. Observations:                      3640   AIC:                         1.663e+04
Df Residuals:                          3633   BIC:                         1.667e+04
Df Model:                                 6                                         
Covariance Type:                    cluster                                         
                  coef    std err          z      P>|z|      [0.025      

'tradeflow_baci'

In [12]:
# First we try a simple Gravity Trade implementation, to show the validity of the data in our dataset.
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]


# We get the standards for this kind of modelling:
columns = [
    "year",
    "tradeflow_baci",
    "gdp_o", "gdp_d",
    "distw_arithmetic",
    "contig",
    "comlang_off",
    "dyad"
]


train_df = train_df[columns]
test_df = test_df[columns]

train_df.dropna()
test_df.dropna()


# As per usual, we want to look at log-trade. 
train_df["baci_log_trade"] = np.log(train_df["tradeflow_baci"])
test_df["baci_log_trade"]  = np.log(test_df["tradeflow_baci"])

train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

X_train = train_df[
    ["log_gdp_o", "log_gdp_d", "log_distw", "contig", "comlang_off", "year"]
]
y_train = train_df["baci_log_trade"]

X_test = test_df[
    ["log_gdp_o", "log_gdp_d", "log_distw", "contig", "comlang_off", "year"]
]
y_test = test_df["baci_log_trade"]

model = LinearRegression()
model.fit(X_train, y_train)


# Finally we can get some workable metrics from this regression:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Out-of-sample RMSE:", rmse)
print("Out-of-sample R²:", r2)


X = sm.add_constant(X_train)
ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                             cov_kwds={"groups": train_df["dyad"]})

print(ols.summary())

Out-of-sample RMSE: 2.417983855373955
Out-of-sample R²: 0.3719398067722376
                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.384
Model:                            OLS   Adj. R-squared:                  0.383
Method:                 Least Squares   F-statistic:                     58.30
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           6.82e-29
Time:                        10:39:31   Log-Likelihood:                -8307.6
No. Observations:                3640   AIC:                         1.663e+04
Df Residuals:                    3633   BIC:                         1.667e+04
Df Model:                           6                                         
Covariance Type:              cluster                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------

### Linear Regression on Conflict Data

We will try to establish a baseline using the conflict data as well. This should be a combination of the trade model above along with a subset of conflict data (as too many regressors will break the model) 

In [13]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

# We can get all the ACLED conflict columns
columns = [
    "year",
    'disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities',

    "distw_arithmetic", "tradeflow_baci", "contig", "comlang_off", "dyad"
]


train_df = train_df[columns]
test_df = test_df[columns]

train_df.dropna()
test_df.dropna()


# As per usual, we want to look at log-trade. 
train_df["baci_log_trade"] = np.log(train_df["tradeflow_baci"])
test_df["baci_log_trade"]  = np.log(test_df["tradeflow_baci"])


train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])
test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

X_train = train_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_train = train_df["baci_log_trade"]

X_test = test_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_test = test_df["baci_log_trade"]

model = LinearRegression()
model.fit(X_train, y_train)


# Finally we can get some workable metrics from this regression:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Out-of-sample RMSE:", rmse)
print("Out-of-sample R²:", r2)


X = sm.add_constant(X_train)
ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                             cov_kwds={"groups": train_df["dyad"]})

print(ols.summary())

Out-of-sample RMSE: 7.020181690167137
Out-of-sample R²: -4.29408828625595
                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.227
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     13.99
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           3.72e-22
Time:                        10:39:35   Log-Likelihood:                -8720.7
No. Observations:                3640   AIC:                         1.750e+04
Df Residuals:                    3611   BIC:                         1.768e+04
Df Model:                          28                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
------------------------

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 31, but rank is 28
  warnings.warn('covariance of constraints does not have full '


In [14]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

# We can get all the ACLED conflict columns
columns = [
    "year",
    'disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities',

    "distw_arithmetic", "tradeflow_baci", "contig", "comlang_off", "dyad"
]


train_df = train_df[columns]
test_df = test_df[columns]

train_df = train_df.dropna()
test_df = test_df.dropna()


# As per usual, we want to look at log-trade. 
train_df["baci_log_trade"] = np.log1p(train_df["tradeflow_baci"])
test_df["baci_log_trade"]  = np.log1p(test_df["tradeflow_baci"])


train_df["log_distw"]  = np.log1p(train_df["distw_arithmetic"])
test_df["log_distw"]  = np.log1p(test_df["distw_arithmetic"])

X_train = train_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_train = train_df["baci_log_trade"]

X_test = test_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_test = test_df["baci_log_trade"]

model = LinearRegression()
model.fit(X_train, y_train)


# Finally we can get some workable metrics from this regression:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Out-of-sample RMSE:", rmse)
print("Out-of-sample R²:", r2)


X = sm.add_constant(X_train)
ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                             cov_kwds={"groups": train_df["dyad"]})

print(ols.summary())

Out-of-sample RMSE: 6.9165191926525456
Out-of-sample R²: -4.366948940443006
                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.232
Model:                            OLS   Adj. R-squared:                  0.226
Method:                 Least Squares   F-statistic:                     14.00
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           3.64e-22
Time:                        10:39:39   Log-Likelihood:                -8603.5
No. Observations:                3640   AIC:                         1.726e+04
Df Residuals:                    3611   BIC:                         1.744e+04
Df Model:                          28                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 31, but rank is 28
  warnings.warn('covariance of constraints does not have full '


In [15]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score


# ==============================================================================
# 1. DEFINER KONFLIKT-FEATURES DYNAMISK VHA. PRÆFIKS
# ==============================================================================
disorder_cols    = [c for c in df.columns if str(c).startswith('disorder_')]
event_cols       = [c for c in df.columns if str(c).startswith('event_')]
perpetrator_cols = [c for c in df.columns if str(c).startswith('perpetrator_')]
target_cols      = [c for c in df.columns if str(c).startswith('target_')]

all_acled_cols = disorder_cols + event_cols + perpetrator_cols + target_cols

# Lav en ren arbejdskopi af df, så originalen ikke røres
df_mod = df.copy()


# ==============================================================================
# 2. VALIDÉR AT TIDSSERIEN ER KONTINUERT PER DYAD (ingen huller i årsrækken)
#    Shift(1) på en hullede tidsserie kobler forkerte år sammen —
#    f.eks. 2012-handel med 2010-konflikter i stedet for 2011-konflikter.
# ==============================================================================
df_mod = df_mod.sort_values(by=["dyad", "year"])

year_gaps = df_mod.groupby("dyad")["year"].diff()
n_gaps    = (year_gaps.dropna() > 2).sum()

if n_gaps > 0:
    print(
        f"⚠️  Advarsel: {n_gaps} hullede årsovergange fundet på tværs af dyader.\n"
        f"   Lag(1) kobler muligvis forkerte år. Overvej at udfylde hullerne\n"
        f"   med NaN-rækker eller filtrere de berørte dyader fra.\n"
    )
else:
    print("✅ Tidsserier er kontinuerte — lag(1) er korrekt.\n")


# ==============================================================================
# 3. SKAB ÉT ÅRS LAG FOR KONFLIKTDATA PER DYAD
#    Vi "skubber" konfliktvariablene ét år fremad inden for hver dyad,
#    så model-inputtet er: konflikter(t-1) → handel(t).
# ==============================================================================
df_mod[all_acled_cols] = df_mod.groupby("dyad")[all_acled_cols].shift(2)

# Det første år per dyad har ingen forgænger og får NaN — disse rækker
# fjernes *ikke* her globalt, men per model inde i run_baseline_model().
# Det sikrer at hver model kun dropper rækker relevante for sine egne features.


# ==============================================================================
# 4. KLARGØR GRAVITY FEATURES OG TARGET (log-transformationer)
#    Nul-handel erstattes med NaN (undgår -inf ved log) og rækker med 
#    manglende GDP eller distance droppes på tværs af alle modeller.
# ==============================================================================
df_mod["baci_log_trade"] = np.log(df_mod["tradeflow_baci"].replace(0, np.nan))
df_mod["log_gdp_o"]      = np.log(df_mod["gdp_o"])
df_mod["log_gdp_d"]      = np.log(df_mod["gdp_d"])
df_mod["log_distw"]      = np.log(df_mod["distw_arithmetic"])

# Tjek for -inf i distw (afstand = 0 ville være et dataproblem)
if np.isinf(df_mod["log_distw"]).any():
    print("⚠️  Advarsel: log_distw indeholder -inf (distw_arithmetic = 0 for én eller flere rækker).")

gravity_cols = ["log_gdp_o", "log_gdp_d", "log_distw"]
# Tilføj ["contig", "comlang_off"] til listen ovenfor hvis du vil have dem med.

# Drop rækker med manglende gravity-variable — disse er fælles for alle modeller
# og kan fjernes globalt uden at påvirke modelsammenlignelighed.
df_mod = df_mod.dropna(subset=gravity_cols + ["baci_log_trade"])


# ==============================================================================
# 5. FUNKTION TIL AT BYGGE OG EVALUERE EN ENKELT MODEL
#    dropna på konflikt-features sker inde i funktionen, så hvert modelsæt
#    kun mister de observationer der mangler i netop dets konfliktkolonner.
# ==============================================================================
def run_baseline_model(conflict_features: list[str], model_name: str) -> dict:
    """
    Bygger og evaluerer én OLS gravity-model med laggede konfliktvariable.

    Parametre
    ----------
    conflict_features : list[str]
        Kolonnenavne for den pågældende ACLED-kategori (allerede lagget i df_mod).
    model_name : str
        Menneskevenlig modelbetegnelse til output.

    Returnerer
    ----------
    dict med nøglerne:
        model_name, n_train, n_test, rmse, r2, ols_results
    """
    print(f"\n{'='*80}")
    print(f"   {model_name}")
    print(f"{'='*80}\n")

    features = gravity_cols + conflict_features

    # --- Drop NaN kun for de relevante conflict_features (FIX #1) ---
    df_clean = df_mod.dropna(subset=features)

    # Train/test split ved 2016 — samme grænse som tidligere
    train_df = df_clean[df_clean["year"] <= 2016]
    test_df  = df_clean[df_clean["year"] > 2016]

    X_train = train_df[features]
    y_train = train_df["baci_log_trade"]
    X_test  = test_df[features]
    y_test  = test_df["baci_log_trade"]

    print(f"Observationer — træning: {len(train_df):,}  |  test: {len(test_df):,}\n")

    # ----------------------------------------------------------------
    # SKLEARN — bruges til out-of-sample RMSE og R²
    # ----------------------------------------------------------------
    sk_model = LinearRegression()
    sk_model.fit(X_train, y_train)
    y_pred = sk_model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"ML  →  Out-of-sample RMSE : {rmse:.4f}")
    print(f"ML  →  Out-of-sample R²   : {r2:.4f}\n")

    # ----------------------------------------------------------------
    # STATSMODELS OLS — bruges til inferens med clustered SE på dyad
    # ----------------------------------------------------------------
    X_train_sm = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X_train_sm).fit(
        cov_type="cluster",
        cov_kwds={"groups": train_df["dyad"]}
    )
    print(ols.summary())

    return {
        "model_name" : model_name,
        "n_train"    : len(train_df),
        "n_test"     : len(test_df),
        "rmse"       : rmse,
        "r2"         : r2,
        "ols_results": ols,
    }


# ==============================================================================
# 6. KØR DE 4 MODELLER OG SAML RESULTATER
# ==============================================================================
model_specs = [
    (disorder_cols,    "1. Linear Regression — Disorder"),
    (event_cols,       "2. Linear Regression — Event"),
    (perpetrator_cols, "3. Linear Regression — Perpetrator"),
    (target_cols,      "4. Linear Regression — Target"),
]

results = [run_baseline_model(cols, name) for cols, name in model_specs]


# ==============================================================================
# 7. SAMMENLIGNINGSTABEL PÅ TVÆRS AF MODELLER
# ==============================================================================
comparison = pd.DataFrame(
    [
        {
            "Model"   : r["model_name"],
            "N train" : r["n_train"],
            "N test"  : r["n_test"],
            "RMSE"    : round(r["rmse"], 4),
            "R²"      : round(r["r2"],   4),
        }
        for r in results
    ]
).set_index("Model")

print("\n" + "="*80)
print("   SAMMENLIGNING — OUT-OF-SAMPLE PERFORMANCE (år > 2016)")
print("="*80)
print(comparison.to_string())
print("="*80 + "\n")

✅ Tidsserier er kontinuerte — lag(1) er korrekt.


   1. Linear Regression — Disorder

Observationer — træning: 3,458  |  test: 910

ML  →  Out-of-sample RMSE : 2.6212
ML  →  Out-of-sample R²   : 0.2619

                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                  0.366
Method:                 Least Squares   F-statistic:                     45.15
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           8.73e-27
Time:                        10:39:44   Log-Likelihood:                -7990.8
No. Observations:                3458   AIC:                         1.600e+04
Df Residuals:                    3450   BIC:                         1.605e+04
Df Model:                           7                                         
Covariance Type:              cluster                                         
      

In [23]:
df_mod["log_conflict_total"] = np.log1p(df_mod["conflict_total"])

run_baseline_model(["log_conflict_total"], 
                   "5. Linear Regression — Log Conflict Intensity")


   5. Linear Regression — Log Conflict Intensity

Observationer — træning: 3,640  |  test: 910

ML  →  Out-of-sample RMSE : 2.4835
ML  →  Out-of-sample R²   : 0.3375

                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.351
Model:                            OLS   Adj. R-squared:                  0.351
Method:                 Least Squares   F-statistic:                     68.02
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           2.17e-26
Time:                        11:12:22   Log-Likelihood:                -8401.6
No. Observations:                3640   AIC:                         1.681e+04
Df Residuals:                    3635   BIC:                         1.684e+04
Df Model:                           4                                         
Covariance Type:              cluster                                         
                         coef    std err  

{'model_name': '5. Linear Regression — Log Conflict Intensity',
 'n_train': 3640,
 'n_test': 910,
 'rmse': 2.4834863813758377,
 'r2': 0.3374509425218698,
 'ols_results': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x137fe8810>}